
## 03: Materialize Gold in Unity Catalog
Reads silver_player_events (UC), applies build_player_daily and
build_game_health_daily from gold.py, writes two UC managed tables. Closes the
lineage graph bronze -> silver -> gold for the player_events branch. Late-data
horizon and replaceWhere are proven locally on 50M; here we only need the
lineage edges, so this is a clean aggregate over the subset.

In [0]:
# %%
import os
from src.ingestion.gold import build_player_daily, build_game_health_daily

CATALOG = "workspace"
SCHEMA = "telemetry"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_player_events"
GOLD_PLAYER_DAILY = f"{CATALOG}.{SCHEMA}.gold_player_daily"
GOLD_GAME_HEALTH_DAILY = f"{CATALOG}.{SCHEMA}.gold_game_health_daily"

# Read Silver by NAME so UC records it as upstream of both Gold tables. Same
# reason as notebook 02: a path read would not attach to the lineage graph.
silver = spark.table(SILVER_TABLE)
print("silver rows:", silver.count())

In [0]:
# %%
# Same pure builders as local. They group on Silver's canonical event_date column
# (never re-derived), so the phantom-31 bug cannot reappear here. gold.py did not
# change between local and cloud: that is the payoff of pure functions.
player_daily = build_player_daily(silver)
game_health_daily = build_game_health_daily(silver)
print("player_daily rows:", player_daily.count())
print("game_health_daily rows:", game_health_daily.count())

In [0]:
# %%
# saveAsTable, partitioned by event_date, overwrite for idempotency. Two Gold
# tables reading one Silver table is what draws the bronze -> silver -> gold fan
# in the catalog.
(
    player_daily.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("event_date")
    .saveAsTable(GOLD_PLAYER_DAILY)
)
print("wrote", GOLD_PLAYER_DAILY)

(
    game_health_daily.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("event_date")
    .saveAsTable(GOLD_GAME_HEALTH_DAILY)
)
print("wrote", GOLD_GAME_HEALTH_DAILY)

In [0]:
# %%
# Confirm all four tables now live in UC: bronze, silver, and two gold.
spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").show(truncate=False)